<a href="https://colab.research.google.com/github/TechnoAceX/Fake_News_Detection/blob/main/Fake_News_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# %% [Install Dependencies] (Run First)
!pip install gym pygame pyvirtualdisplay > /dev/null
!apt-get install -y xvfb python-opengl ffmpeg > /dev/null

# %% [Imports]
import gym
import numpy as np
import matplotlib.pyplot as plt
from IPython import display
from pyvirtualdisplay import Display

# Setup virtual display for Colab
Display(visible=0, size=(400, 300)).start()

# %% [Environment Setup]
env = gym.make('FrozenLake-v1', is_slippery=False, render_mode='rgb_array')
Q = np.zeros((env.observation_space.n, env.action_space.n))

# Training parameters
EPISODES = 1500
MAX_STEPS = 100
LEARNING_RATE = 0.81
GAMMA = 0.96
epsilon = 0.9
rewards = []

# %% [Training Loop]
for episode in range(EPISODES):
    state = env.reset()[0]  # Get state from tuple
    episode_reward = 0

    for _ in range(MAX_STEPS):
        # Epsilon-greedy action selection
        if np.random.uniform(0, 1) < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(Q[state, :])

        next_state, reward, terminated, truncated, _ = env.step(action)

        # Q-value update
        Q[state, action] += LEARNING_RATE * (
            reward + GAMMA * np.max(Q[next_state, :]) - Q[state, action]
        )

        state = next_state
        episode_reward += reward

        if terminated or truncated:
            break

    rewards.append(episode_reward)
    epsilon = max(0.01, epsilon * 0.995)

    if episode % 100 == 0:
        avg_reward = np.mean(rewards[-100:])
        print(f"Episode {episode}, Avg Reward: {avg_reward:.2f}")

# %% [Results]
print("\nTrained Q-table:")
print(Q)

# Plot training progress
plt.figure(figsize=(12, 6))
plt.plot(np.convolve(rewards, np.ones(100)/100, mode='valid'))
plt.title("Average Reward per 100 Episodes")
plt.xlabel("Episodes")
plt.ylabel("Success Rate")
plt.show()

# %% [Test Agent]
def test_agent():
    test_env = gym.make('FrozenLake-v1', is_slippery=False, render_mode='rgb_array')
    state = test_env.reset()[0]
    done = False

    while not done:
        action = np.argmax(Q[state, :])
        state, reward, done, truncated, _ = test_env.step(action)

        # Render frame
        plt.imshow(test_env.render())
        plt.axis('off')
        display.display(plt.gcf())
        display.clear_output(wait=True)
        time.sleep(0.5)

        if done:
            print(f"Outcome: {'Success!' if reward == 1 else 'Failure'}")
            break

    test_env.close()

test_agent()

E: Unable to locate package python-opengl


AttributeError: module 'numpy' has no attribute 'bool8'

**1. Importing Libraries**

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
import string

# Scikit-learn modules
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression

# Machine Learning Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    StackingClassifier
)


**2. Loading the Dataset**

In [ ]:
data_fake = pd.read_csv('Fake.csv')
data_true = pd.read_csv('True.csv')

**3. Previewing Data**

In [ ]:
data_fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [ ]:
data_true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


**4. Labeling the Data**

In [ ]:
data_fake["class"] = 0
data_true["class"] = 1

**5. Checking Dataset Shapes**

In [ ]:
data_fake.shape, data_true.shape

((23481, 5), (21417, 5))

In [ ]:
# Extract last 10 rows for manual testing
data_fake_manual_testing = data_fake.tail(10)
data_true_manual_testing = data_true.tail(10)

# Drop the last 10 rows from the original datasets using a loop
for i in range(13654, 13640 - 1, -1):
    data_fake.drop([i], axis=0, inplace=True)

for i in range(14179, 14169 - 1, -1):
    data_true.drop([i], axis=0, inplace=True)

In [ ]:
data_fake.shape, data_true.shape

((23466, 5), (21406, 5))

In [ ]:
data_fake_manual_testing['class'] = 0
data_true_manual_testing['class'] = 1

<ipython-input-12-90008d39c97b>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_fake_manual_testing['class'] = 0
<ipython-input-12-90008d39c97b>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_true_manual_testing['class'] = 1


In [ ]:
data_fake_manual_testing.head(10)

,title,text,subject,date,class
23471,Seven Iranians freed in the prisoner swap have...,"21st Century Wire says This week, the historic...",Middle-east,"January 20, 2016",0
23472,#Hashtag Hell & The Fake Left,By Dady Chery and Gilbert MercierAll writers ...,Middle-east,"January 19, 2016",0
23473,Astroturfing: Journalist Reveals Brainwashing ...,Vic Bishop Waking TimesOur reality is carefull...,Middle-east,"January 19, 2016",0
23474,The New American Century: An Era of Fraud,Paul Craig RobertsIn the last years of the 20t...,Middle-east,"January 19, 2016",0
23475,Hillary Clinton: ‘Israel First’ (and no peace ...,Robert Fantina CounterpunchAlthough the United...,Middle-east,"January 18, 2016",0
23476,McPain: John McCain Furious That Iran Treated ...,21st Century Wire says As 21WIRE reported earl...,Middle-east,"January 16, 2016",0
23477,JUSTICE? Yahoo Settles E-mail Privacy Class-ac...,21st Century Wire says It s a familiar theme. ...,Middle-east,"January 16, 2016",0
23478,Sunnistan: US and Allied ‘Safe Zone’ Plan to T...,Patrick Henningsen 21st Century WireRemember ...,Middle-east,"January 15, 2016",0
23479,How to Blow $700 Million: Al Jazeera America F...,21st Century Wire says Al Jazeera America will...,Middle-east,"January 14, 2016",0
23480,10 U.S. Navy Sailors Held by Iranian Military ...,21st Century Wire says As 21WIRE predicted in ...,Middle-east,"January 12, 2016",0


In [ ]:
data_fake_manual_testing.head(10)

,title,text,subject,date,class
23471,Seven Iranians freed in the prisoner swap have...,"21st Century Wire says This week, the historic...",Middle-east,"January 20, 2016",0
23472,#Hashtag Hell & The Fake Left,By Dady Chery and Gilbert MercierAll writers ...,Middle-east,"January 19, 2016",0
23473,Astroturfing: Journalist Reveals Brainwashing ...,Vic Bishop Waking TimesOur reality is carefull...,Middle-east,"January 19, 2016",0
23474,The New American Century: An Era of Fraud,Paul Craig RobertsIn the last years of the 20t...,Middle-east,"January 19, 2016",0
23475,Hillary Clinton: ‘Israel First’ (and no peace ...,Robert Fantina CounterpunchAlthough the United...,Middle-east,"January 18, 2016",0
23476,McPain: John McCain Furious That Iran Treated ...,21st Century Wire says As 21WIRE reported earl...,Middle-east,"January 16, 2016",0
23477,JUSTICE? Yahoo Settles E-mail Privacy Class-ac...,21st Century Wire says It s a familiar theme. ...,Middle-east,"January 16, 2016",0
23478,Sunnistan: US and Allied ‘Safe Zone’ Plan to T...,Patrick Henningsen 21st Century WireRemember ...,Middle-east,"January 15, 2016",0
23479,How to Blow $700 Million: Al Jazeera America F...,21st Century Wire says Al Jazeera America will...,Middle-east,"January 14, 2016",0
23480,10 U.S. Navy Sailors Held by Iranian Military ...,21st Century Wire says As 21WIRE predicted in ...,Middle-east,"January 12, 2016",0


**6. Combining Both Datasets**

In [ ]:
data_merge = pd.concat([data_fake, data_true], axis = 0)
data_merge.head(10)

,title,text,subject,date,class
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",0
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",0
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",0
5,Racist Alabama Cops Brutalize Black Boy While...,The number of cases of cops brutalizing and ki...,News,"December 25, 2017",0
6,"Fresh Off The Golf Course, Trump Lashes Out A...",Donald Trump spent a good portion of his day a...,News,"December 23, 2017",0
7,Trump Said Some INSANELY Racist Stuff Inside ...,In the wake of yet another court decision that...,News,"December 23, 2017",0
8,Former CIA Director Slams Trump Over UN Bully...,Many people have raised the alarm regarding th...,News,"December 22, 2017",0
9,WATCH: Brand-New Pro-Trump Ad Features So Muc...,Just when you might have thought we d get a br...,News,"December 21, 2017",0


In [ ]:
data_merge.columns

Index(['title', 'text', 'subject', 'date', 'class'], dtype='object')

In [ ]:
data = data_merge.drop(['title', 'subject', 'date'], axis = 1)

In [ ]:
data.isnull().sum()

,0
text,0
class,0


In [ ]:
data = data.sample(frac = 1)

In [ ]:
data.head()

,text,class
18095,ANKARA (Reuters) - Turkish President Tayyip Er...,1
20376,LONDON (Reuters) - British lawmakers on Tuesda...,1
1178,One of the GOP s biggest donors who used to be...,0
6032,"Ana Navarro, a Republican strategist, commenta...",0
16422,But the media s concerned Trump is the threat ...,0


**7. Data Cleaning and Index Reset**

In [ ]:
data.reset_index(inplace = True)
data.drop(['index'], axis = 1, inplace = True)
data.columns

Index(['text', 'class'], dtype='object')

In [ ]:
data.head()

,text,class
0,ANKARA (Reuters) - Turkish President Tayyip Er...,1
1,LONDON (Reuters) - British lawmakers on Tuesda...,1
2,One of the GOP s biggest donors who used to be...,0
3,"Ana Navarro, a Republican strategist, commenta...",0
4,But the media s concerned Trump is the threat ...,0


**8. Text Preprocessing Function**

In [ ]:
def wordopt(text):
    text = text.lower()  # Convert text to lowercase
    text = re.sub(r'\[.*?\]', '', text)  # Remove text inside square brackets
    text = re.sub(r'\W', ' ', text)  # Remove non-word characters
    text = re.sub(r'https?://\S+|www\.\S+', '', text)  # Remove URLs
    text = re.sub(r'<.*?>+', '', text)  # Remove HTML tags
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)  # Remove punctuation
    text = re.sub(r'\n', ' ', text)  # Remove newlines
    text = re.sub(r'\w*\d\w*', '', text)  # Remove words containing digits
    return text

In [ ]:
data['text'] = data['text'].apply(wordopt)

**10. Splitting Data into Training and Testing Sets**

In [ ]:
x = data['text']
y = data['class']

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size = 0.25)

# Text Vectorization using TF-IDF

In [ ]:
# Feature extraction using TF-IDF
vectorization = TfidfVectorizer()
xv_train = vectorization.fit_transform(x_train)
xv_test = vectorization.transform(x_test)

# Decision Tree Classifier

In [ ]:
# Decision Tree Classifier
DT = DecisionTreeClassifier()
DT.fit(xv_train, y_train)
pred_DT = DT.predict(xv_test)

In [ ]:
DT.score(xv_test, y_test)

0.9951863077197362

In [ ]:
print(classification_report(y_test, pred_DT))

              precision    recall  f1-score   support

           0       0.99      1.00      1.00      5888
           1       1.00      0.99      0.99      5330

    accuracy                           1.00     11218
   macro avg       1.00      1.00      1.00     11218
weighted avg       1.00      1.00      1.00     11218



# Gradient Boosting Classifier

In [ ]:
# Gradient Boosting Classifier
GB = GradientBoostingClassifier(random_state=0)
GB.fit(xv_train, y_train)
pred_GB = GB.predict(xv_test)

In [ ]:
GB.score(xv_test, y_test)

0.9953645926190051

In [ ]:
print(classification_report(y_test, pred_GB))

              precision    recall  f1-score   support

           0       1.00      0.99      1.00      5888
           1       0.99      1.00      1.00      5330

    accuracy                           1.00     11218
   macro avg       1.00      1.00      1.00     11218
weighted avg       1.00      1.00      1.00     11218



# Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest Classifier
RF = RandomForestClassifier(random_state=0)
RF.fit(xv_train, y_train)
pred_RF = RF.predict(xv_test)

In [ ]:
RF.score(xv_test, y_test)

0.9893920484934926

In [ ]:
print(classification_report(y_test, pred_RF))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5888
           1       0.99      0.99      0.99      5330

    accuracy                           0.99     11218
   macro avg       0.99      0.99      0.99     11218
weighted avg       0.99      0.99      0.99     11218



# Accuracy and Classification Report

In [ ]:
# Print accuracy and classification reports
print("Decision Tree Accuracy:", accuracy_score(y_test, pred_DT))
print(classification_report(y_test, pred_DT))

print("Gradient Boosting Accuracy:", accuracy_score(y_test, pred_GB))
print(classification_report(y_test, pred_GB))

print("Random Forest Accuracy:", accuracy_score(y_test, pred_RF))
print(classification_report(y_test, pred_RF))

Decision Tree Accuracy: 0.9951863077197362
              precision    recall  f1-score   support

           0       0.99      1.00      1.00      5888
           1       1.00      0.99      0.99      5330

    accuracy                           1.00     11218
   macro avg       1.00      1.00      1.00     11218
weighted avg       1.00      1.00      1.00     11218

Gradient Boosting Accuracy: 0.9953645926190051
              precision    recall  f1-score   support

           0       1.00      0.99      1.00      5888
           1       0.99      1.00      1.00      5330

    accuracy                           1.00     11218
   macro avg       1.00      1.00      1.00     11218
weighted avg       1.00      1.00      1.00     11218

Random Forest Accuracy: 0.9893920484934926
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5888
           1       0.99      0.99      0.99      5330

    accuracy                           0.99     1

# Manual Testing Function

In [ ]:
# Function to output label based on prediction
def output_label(n):
    """
    Returns a human-readable label based on the model's prediction.
    """
    return "Fake News" if n == 0 else "Not A Fake News"

# Function for manual testing of a given news article
def manual_testing(news):
    """
    Takes a news article as input and predicts whether it is fake or real
    using multiple models.
    """
    testing_news = {"text": [news]}
    new_def_test = pd.DataFrame(testing_news)
    new_def_test["text"] = new_def_test["text"].apply(wordopt)

    new_xv_test = vectorization.transform(new_def_test["text"])

    # Predictions from different models
    pred_LR = LR.predict(new_xv_test)
    pred_DT = DT.predict(new_xv_test)
    pred_GB = GB.predict(new_xv_test)
    pred_RF = RF.predict(new_xv_test)

    # Print results
    print("\n\nPredictions:")
    print(f"Logistic Regression: {output_label(pred_LR[0])}")
    print(f"Decision Tree: {output_label(pred_DT[0])}")
    print(f"Gradient Boosting: {output_label(pred_GB[0])}")
    print(f"Random Forest: {output_label(pred_RF[0])}")

# Hybrid Mode

In [ ]:
# 1️⃣ Load the datasets
df_fake = pd.read_csv('Fake.csv')
df_true = pd.read_csv('True.csv')

# Add labels
df_fake['label'] = 0  # Fake news
df_true['label'] = 1  # True news

# Combine datasets
df = pd.concat([df_fake, df_true])

# 🛠 Reduce dataset size for faster training (adjust as needed)
df = df.sample(5000, random_state=42)

# 2️⃣ Remove missing values
df = df.dropna(subset=["text"])

# 3️⃣ Prepare features (X) and labels (y)
X = df["text"]
y = df["label"]

# 4️⃣ Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5️⃣ TF-IDF Vectorization (Reduce max_features for efficiency)
vectorizer = TfidfVectorizer(max_features=3000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# 🔥 Convert sparse TF-IDF matrix to dense format
X_train_dense = X_train_tfidf.toarray()
X_test_dense = X_test_tfidf.toarray()

# 6️⃣ Define base models with optimized parameters
models = {
    'dt': DecisionTreeClassifier(max_depth=10),
    'rf': RandomForestClassifier(n_estimators=10, warm_start=True, n_jobs=1, random_state=0),
    'gb': HistGradientBoostingClassifier(random_state=0)
}

print("\n🔹 Training base models...\n")

for name, model in models.items():
    model.fit(X_train_dense, y_train)  # Use dense matrix
    y_pred = model.predict(X_test_dense)
    acc = accuracy_score(y_test, y_pred)
    print(f"✅ Accuracy of {name}: {acc:.4f}")

# 7️⃣ Define Stacking Classifier with n_jobs=1
base_models = [(name, model) for name, model in models.items()]
meta_model = LogisticRegression(max_iter=300, solver='lbfgs')

stacked_model = StackingClassifier(estimators=base_models, final_estimator=meta_model, n_jobs=1)

# 8️⃣ Train Stacking Model
print("\n🚀 Training the stacking classifier...\n")
stacked_model.fit(X_train_dense, y_train)  # Use dense matrix
print("✅ Stacking model training completed!\n")

# 9️⃣ Evaluate Stacking Model
y_pred_stacked = stacked_model.predict(X_test_dense)

# 🔟 Print results
print("✅ Stacking Classifier Accuracy:", accuracy_score(y_test, y_pred_stacked))
print("\n🔹 Classification Report:\n", classification_report(y_test, y_pred_stacked))


🔹 Training base models...

✅ Accuracy of dt: 0.9870
✅ Accuracy of rf: 0.9720
✅ Accuracy of gb: 0.9930

🚀 Training the stacking classifier...

✅ Stacking model training completed!

✅ Stacking Classifier Accuracy: 0.993

🔹 Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.99      0.99       524
           1       0.99      1.00      0.99       476

    accuracy                           0.99      1000
   macro avg       0.99      0.99      0.99      1000
weighted avg       0.99      0.99      0.99      1000



# Example manual testing

In [ ]:
def manual_testing(news):
    # Use the trained vectorizer to transform the input
    new_xv_test = vectorizer.transform([news])  # Ensure it uses the same vectorizer
    new_xv_test_dense = new_xv_test.toarray()  # Convert to dense format

    print("\n🔍 Predictions:")
    print(f"✅ Decision Tree: {'Fake News' if models['dt'].predict(new_xv_test_dense)[0] == 0 else 'Not Fake News'}")
    print(f"✅ Random Forest: {'Fake News' if models['rf'].predict(new_xv_test_dense)[0] == 0 else 'Not Fake News'}")
    print(f"✅ Gradient Boosting: {'Fake News' if models['gb'].predict(new_xv_test_dense)[0] == 0 else 'Not Fake News'}")
    print(f"✅ Stacking Model: {'Fake News' if stacked_model.predict(new_xv_test_dense)[0] == 0 else 'Not Fake News'}")

# 📝 User input for testing
news_input = input("\n📝 Enter a news article for testing: ")
manual_testing(news_input)


🔍 Predictions:
✅ Decision Tree: Fake News
✅ Random Forest: Fake News
✅ Gradient Boosting: Fake News
✅ Stacking Model: Fake News


SEOUL (Reuters) - North Korean leader Kim Jong Un guided a launch of its Hwasong-12 intermediate-range ballistic missile on Tuesday in a drill to counter the joint military exercises by South Korean and U.S. militaries, the North s official KCNA news agency said on Wednesday.  The current ballistic rocket launching drill like a real war is the first step of the military operation of the KPA in the Pacific and a meaningful prelude to containing Guam,  KCNA quoted Kim as saying. KPA stands for the Korean People s Army, the North s military. North Korea threatened to fire four Hwasong-12 missiles into the sea near the U.S. Pacific territory of Guam earlier this month after U.S. President Donald Trump said the North would face  fire and fury  if it threatened the United States.
